# DNAStream v0.1.1-beta Tutorial

## Installation from the command line

```bash
# optional but recommended
conda create -n dnastream python=3.11
conda activate dnastream

pip install "dnastream @ git+https://github.com/VanLoo-lab/DNAStream.git"
```

Verify the installation:

```bash
python -c "import dnastream; from dnastream import DNAStream; print(dnastream.__version__)"
```

## Direct installation into the kernel
if you haven't installed into your environment/kernel already, run the below block.

In [ ]:

%pip install "dnastream @ git+https://github.com/VanLoo-lab/DNAStream.git"

## Verify the installation 
Check that you can import the `dnsastream` package and are running the correct version (currently 0.1.1b0).

In [ ]:

import dnastream; print(dnastream.__version__)


## Import packages for the tutorial

In [ ]:
from pathlib import Path
import tempfile

#create a temp directory and temp file 
tmpdir = Path(tempfile.mkdtemp(prefix="dnastream_tutorial_"))
myfile = tmpdir / "temp.h5"

#import the main interface class DNAStream
from dnastream import DNAStream

# How to create and stream a DNAStream file

`DNAstream.create(...)` function initializes the HDF5-file and creates all built-in datasets.  

It returns a `DNAStream` object that provides an interface to the HDF5 file. 

When finished streaming the data, `close()` is used to safely close the stream in order to avoid file corruption.


### CLI tool equivalent command
```bash
dnastream create -f {fname} -p patientX --verbose
```


In [ ]:

# Create the DNAStream file 
ds = DNAStream.create(path=myfile, patient_id="patientX", verbose=True)
ds.close()




## Opening a stream to the DNAStream file
There are two available modes for streaming a DNAStream file, 'r' and 'r+'.   
- 'r' allows read access only and prevents modifications to the DNAStream file (default)
- 'r+' allows read/write access to facilitate modfications to the DNAStream file

The `DNAStream.open(...)` function is used to open the stream to the file and returns a `DNAStream` object to interface with data.

It is **recommended** to stream a DNAStream file with a context manager. This ensure that in the event of errors or exceptions, the stream is automatically closed to print file corruption.

In [ ]:
#verbose can be used to print additional messages to the console (default is False)
with DNAStream.open(path=myfile, mode="r", verbose=True) as ds:
    print(ds) #print the DNAStream object ds

## What is included currently in a DNAStream?

`DNAStream` currently includes three built-in registries/provenance attributes and an I/O interface attribute
- `sample`  : sample registry
- `variant` : variant registry
- `snp` : SNP registry
- `log` : File modification provenance
- `io` : I/O interface for DNAStream file

They can be accessed as attributes from a `DNAStream` object or by name via the `registry(name)` or `provenance(name)` function within DNAStream.

Additionally, there are two properties:
- `patient_id` : the id of the patient under study
- `mode` : the current streaming mode of the DNAStream interface

In [ ]:
with DNAStream.open(myfile) as ds:
    print(ds.sample) #a reference to the sample registry object
    print(ds.registry("sample")) # a function that returns a reference
    print(ds.log)
    print(ds.provenance("log"))

### Change or update the patient id
patient id is metadata that is stored in the HDF5 file itself.

In [ ]:
with DNAStream.open(myfile, mode="r+") as ds:
    print(f"Current patient id: {ds.patient_id}")
    ds.set_patient_id("patientY")
    # ds.patient_id = "foo" throws an exception
    print(f"New patient id: {ds.patient_id}")

## Registry objects and their role in DNAStream
Registries play a key role in DNAStream because they provide tracking and management of key research entities, such as samples, cells, SNVs, indels, SNPs. 

When multiple analysts are working on a project, it can be difficult to track which samples pass QA and are to be used in downstream analysis or what SNVs are the current active set when multiple SNV callers are run. 

It can be hard to manage metadata and easily format it to streamline bash and workflow scripts/pipelines. They also provide link to various measured data and computed analysis results. 

Formally, a registry is a structured 1-d **append only** dataset (think dataframe) stored in the DNAStream HDF5 file for perpetuity. Any modifications to the registry dataset are logged so registry event histories can be extracted. 

When streaming a DNAStream file, `Registry` objects serve as interfaces to the registry dataset. Via `Registry` objects, one can:   
- `add` new entities for registration
- query the registry to `get` a subset meeting selector critera, 
- `update` the metadata of editable fields
- `activate` entities by `id` or `label`
- `find` all entities by a given `label`
- extract all or part of the registry dataset `to_dataframe`

### The registry spine
Every registry dataset automatically includes the following eight fields:
1. id : this is a unique identifier (UUIDv4) string for every registered entity.
2. label : this an internally generated **non-unique** human-readable string used to identify entities. The schema will specify how this field is generated. 
3. idx : this is a internal index of the entity in the dataset
4. active : this is a boolean indicating if the entity is active. Registry policy is to allow at most one unique label to be active. 
5. created_at : timestamp the entity was registered
6. created_by : user that registered the entity
7. modified_at : timestamp the entity was last modified
8. modified_by : the user that last modified the entity.

*NOTE: Registry policy does not allow modification of registry spine fields.  It also prevents updated any fields used to generate the label field.*


## Add entities to the registry
There are multiple options to register entities. 

Registry items can be added directly to a `Registry` object via the low-level `Registry.add(...)` function from a list of dictionaries, where each dictionary is an entity.  

But there is also an `io` class to help streamline insertation from standard formats ('maf' files) and CSV and TSV files. 

### Low-level insertion
To utilize low-level insertation, first obtain a reference to a 'Registry' object and then call the `add(...)` function on a list of dictionaries.


In [ ]:
#open stream in mode 'r+' for registry dataset modification.
with DNAStream.open(path=myfile, mode="r+") as ds:

    #pointer to the built-in sample registry
    reg = ds.sample
    print(f"Sample registry contains {len(reg)} entities.")

    records = [  
        {"sample_name": "S1", "modality": "bulk"},
        {"sample_name": "S2", "modality": "single-cell"},
    ] #list of dictionaries, each entity is a dictionary if field name as key

    reg.add(records,allow_duplicate_labels=True, 
            defaults={"organism": "human", "center_name": "MDACC"} )

    print(f"After add, sample registry contains {len(reg)} entities.")

## `add` function parameters

There are three key parameters to the `add(...)` function which deal with the fact that internal `labels` are not required to be unique:
- `allow_duplicate_labels` : whether to allow the insertation to proceed with inserting new records that collide with existing registered entities (default is `False`).
- `activate_newest` : in the event of duplicate labels, this specifies whether to activate the new entity upon insertation (and deactivate the old entity) or whether to leave the newly inserted colliding entity deactivated (default is `True`).
- `defaults`: takes in a dictionary, with field name as key and default value as value and inserts a default value for a field for all inserted records.

In [ ]:
#open stream in mode 'r+' for registry dataset modification.
with DNAStream.open(path=myfile, mode="r+") as ds:

    #pointer to the built-in sample registry
    reg = ds.sample
    print(f"Sample registry contains {len(reg)} entities.")

    reg.add(records,
        allow_duplicate_labels=True, 
        activate_newest=True)

    print(f"After add, sample registry contains {len(reg)} entities.")

In [ ]:
with DNAStream.open(myfile, mode="r+") as ds:
       
    ds.variant.add([
        {"chrom": "chr1", "start_pos": 1231, "end_pos": 1232, "ref_allele": "A", "alt_allele": "T"},
    ])

    for snv in ds.variant:
        print(f"SNV id: {snv['id']}, SNV label: {snv['label']}, active: {snv['active']}")

## Key Python special methods
The `Registry` interface implements special Python methods for ease of use:
- `__len__`: get the number of registered entities,
- `__iter__` : iterate through each entity in the registry
- `__get__` : slice the registry using the unique id of an entry
- `__contains__` : check if a unique id is present in the registry 

In [ ]:
with DNAStream.open(myfile) as ds:
    
    nrecords = len(ds.sample) #get the number of entities
    print(f"The 'sample' registry contains {nrecords} entities")

    #iterate through the registry yielding each row as a dictionary
    for record in ds.variant:
        print(record)
        my_id = record["id"]
    
    #slice the variant registry dataset by id
    record =ds.variant[my_id]
    print(record)

    # check if my_id is present in the variant registry dataset
    print(f"id: {my_id} present in variant registry: {my_id in ds.variant}") 

    #check if my_id is present in the sample registry dataset
    print(f"id: {my_id} present in sample registry: {my_id in ds.sample}") 


## Update the metadata of registered entities
Only certain fields in a registry are modifiable. 

Registry datasets are only modifiable via the `Registry.update(...)` function.  

To provide additional guardrails, the unique id of an entity is needed to modify its metadata.  

### Updating modality
In this example, we will look up the unique ids associated with sample 'S1' and then we will change its 'modality' in the registry to 'lcm'.  

We pass update a list of dictionaries, where each dictionary must key contain the key 'id' and any fieldnames with associated values to update.

In [ ]:
#let's change the modality from bulk to lcm.
with DNAStream.open(myfile, mode="r+") as ds:
    print("Before....")
    for entry in ds.sample:
        if entry['label']=="S1":
            print(f"id: {entry['id']}, label: {entry['label']}, modality: {entry['modality']}")
    
    print("After....")
    sample_ids =ds.sample.find_ids(labels="S1")  #looks up the unique ids associated with sample "S1"
    
    rows = [{"id": id, "modality": "lcm"} for id in sample_ids["S1"]]
    ds.sample.update(rows, warn_missing=True)
    for entry in ds.sample:
        if entry['label']=="S1":
            print(f"id: {entry['id']}, label: {entry['label']}, modality: {entry['modality']}")

# Activate and deactivate registered entities

One of the main advantages of registering entities is that all users can view and track which the current set of research entities to include in their downstream analysis. 

`Registry` interfaces provides functionality to activate and deactive entities by 'id' or 'label'.  

**Note:** Caution should be used when activating and deactivating entities by label as there may be multiple labels. Review the DNAStream policy and pay careful attention to function arguments to ensure proper activation occurs.

Suppose sample 'S1' fails QA, lets deactivate all entities associated with sample 'S1'. 

In [ ]:
with DNAStream.open(myfile, mode="r+") as ds:
    print("Before...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")
       
    ds.sample.deactivate_labels(labels="S1")

    print("After...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")

Now, let's reactivate sample "S1" by label. We will use the default value of `activate_newest` to ensure the most recent sample 'S1' entity added to the registry is activated.

In [ ]:
with DNAStream.open(myfile, mode="r+") as ds:
    print("Before...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")
       
    ds.sample.activate_labels(labels="S1", activate_newest=True)

    print("After...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")

Finally, let's suppose that we want to activate the other sample 'S1' instead. We can use its unique id to ensure that entity is activated.  Any other entities with the same label will then be deactivated.

In [ ]:
with DNAStream.open(myfile, mode="r+") as ds:
    print("Before...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")

    id_dict = ds.sample.find_ids(labels="S1")   
    for id in id_dict["S1"]:
        if not ds.sample[id]["active"]:
            my_id = id
    ds.sample.activate_ids(ids = my_id)

    print("After...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")

## Query/export the registry or a subset of the registry to a `pandas.Dataframe`

Users can query or export views of a registry dataset as a `pandas.Dataframe` via two different functions:`to_dataframe(...)` and `get(...)` 

 **Note: these only return copies of the dataset as a dataframe and any modifications of this dataframe will not be reflected in the DNAStream HDF5 file.**


 The `to_dataframe()` function is a convenient way to extract the entire registry, or typical subsets, such as all active entities or only deactivated entities. This is passed through the `mode` parameter.  

In [ ]:
with DNAStream.open(myfile) as ds:
    sample_registry_df = ds.sample.to_dataframe(mode="all") #default
    print(f"Shape:{sample_registry_df.shape}")
    print(type(sample_registry_df))
    print(sample_registry_df)

    active_samples_df = ds.sample.to_dataframe(mode="active_only")
    print(f"Shape:{active_samples_df.shape}")

    non_active_samples_df = ds.sample.to_dataframe(mode="non_active")
    print(f"Shape:{non_active_samples_df.shape}")

### Filtering and queries beyond activation status
However, users may want to make more complex queries to the registry return the records as a `pandas.Dataframe`. 

For this, the `get(...)` function should be used instead of `to_dataframe()`.  The key argument to `get(...)` is selector. `selector` can be list of labels, ids or a callable function that will subset a `pandas.Dataframe`.

In [ ]:
#subset the sample registry by entities with label "S1"
with DNAStream.open(myfile) as ds:

    S1_df = ds.sample.get(selector=["S1"], by="label")
    print(type(S1_df))
    print(S1_df)

## I/O functions
Typically, users will want to register a large number of entities from the output files of other upstream methods, like variant callers (maf files) or via CSV and TSV files.  `DNAStream` provides `io` wrappers to streamline the insertation of entities to built-in registries from files. 

Currently, `DNAStream` offers three `io` functions to add facilitate this:
1. `io.add_variants_from_maf`: add variants to the variant registry from a maf file or a list of maf files.
2. `io.add_snps_from_maf` : add SNPs to the SNP registry from a maf file or a list of maf files.
3. `io.add_samples_from_files` : add samples to the sample registry from a CSV (default) or other delimited file.

Additionally, `DNAStream` offers static functions to parse CSV (`io.parse_csv`, `io.parse_tsv`) or TSV files for preparation in the list of dictionary format required by `Registry.add(...)`.

Includes the parameters for add plus also:
- `column_mapping`: a dictionary mapping column name in file to schema field name, i.e, `{"sample_id": "sample_name"}`
-  `allow_duplicate_source_files`: a unique file identifier is stored in the registry when entities are registered for a source file. By default, DNAStream won't allow you to reinsert entities from the same source file (default: `False`) unless you override this. 



In [ ]:
#First, we will generate some temporary files with example data.

maf_text = "Hugo_Symbol\tChromosome\tStart_Position\tEnd_Position\tReference_Allele\tTumor_Seq_Allele2\tTumor_Sample_Barcode\nTP53\tchr17\t7579472\t7579472\tC\tT\tS1\n"
csv_text = "sample_name,modality\nS3,bulk\nS4,single-cell\n"

with tempfile.NamedTemporaryFile("w", suffix=".maf", delete=False) as maf_fp:
    maf_fp.write(maf_text)
    maf_path = maf_fp.name

with tempfile.NamedTemporaryFile("w", suffix=".csv", delete=False) as csv_fp:
    csv_fp.write(csv_text)
    csv_path = csv_fp.name

maf_path, csv_path

In [ ]:
#add samples from csv file
with DNAStream.open(myfile, mode="r+") as ds:
    print(f"Sample registry started with {len(ds.sample)} entities.")
    ds.io.add_samples_from_files(csv_path)
    print(f"Sample registry now contains {len(ds.sample)} entities.")
    print(ds.sample.to_dataframe())

In [ ]:
with DNAStream.open(myfile, mode="r+") as ds:
    print(f"Variant registry started with {len(ds.variant)} entities.")
    ds.io.add_variants_from_maf(maf_path)
    print(f"Variant registry now contains {len(ds.variant)} entities.")
    print(ds.variant.to_dataframe())

## `Provenance` objects and their role in DNAStream

 DNAStream was designed to facilitate one or more uses to complete many upstream and downstream analyses on mult-modal DNA sequencing data. 
 
 Subsequently, it becomes important to track any modifications to the DNAStream HDF5 file. Provenance datasets are used to track these modifications.  
 
 Currently, there is only one built-in provenance dataset name 'log'.  Similarily to `Registry`, DNAStream provides a specialized `Provenance` interface to a provenance dataset. 

Via `Provenance` objects, one can:   
- extract the provenance log via `to_dataframe`
- iterate through the provenance log dataset
- determine the number of logged modifications


### Provenance log fields

The provenance log contains the the following fields:
- id : unique identifier of the event (UUIDv4)
- timestamp : time and date of when the modification occurred
- user : the user that performed the mofidication
- scope : which interface (DNAStream, IO, Registry, etc) performed the modification
- event : type of modification; one of create, append, modify, designate
- dataset : the dataset in the HDF5 file where the modification occurred
- source : the string name of the function that was used to modify the file
- info : a JSON string that includes amplifying information about the modification


The built-in provenance modfication log can be acesssed in two ways, by attribute or by a `provenance(name)` method.

In [ ]:
with DNAStream.open(myfile) as ds:

    print(ds.log)
    log = ds.provenance("log")
    assert ds.log == log

## Using Python special methods on a `Provenance` object
The `Provenance` interface provides the following built Python special methods:
- `__iter__` : iterate through each logged modification in the provenance dataset
- `__len__` : get the total number of entries in the provenance dataset.

In [ ]:
with DNAStream.open(myfile) as ds:
    nevents = len(ds.log)
    print(f"The modification log contains {nevents} events.")

    for mod in ds.log:
        if mod["event"] == "create":
            print(mod)

## Query/export the provenance log or a subset to a `pandas.Dataframe`.
Users can export views of a proveance dataset as a `pandas.Dataframe` via  the `to_dataframe(...)` function. 

**Note, these only return copies of the dataset as a dataframe and any modifications of this dataframe will not be reflected in the DNAStream HDF5 file.** 

In [ ]:
with DNAStream.open(myfile) as ds:

    log_df = ds.log.to_dataframe()

    print(log_df)